In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
import cv2
import bm3d
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from skimage.restoration import estimate_sigma
from torch.utils.data import DataLoader
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

from admmtor.eprocessing.dataload import ImageDataset
from admmtor.inference.patch_merge_inference import PatchMergeInference
from admmtor.modelbuild.ffdnet import FFDNet
from admmtor.modelbuild.dncnn import DnCNN
from admmtor.modelbuild.denoiser import DivergentRestorer
from admmtor.modelbuild.nafnet import NAFNet
from admmtor.modelbuild.dranet import DRANet
from admmtor.modelbuild.restormer import Restormer
from admmtor.modelbuild.swinir import SwinIR
from admmtor.eprocessing.etransforms import Scale, RandCrop, AddAWGN
from admmtor.emetrics.metrics import SSIMMetric, MSSSIMMetric, PSNRMetric, SCCMetric, UIQMetric, MSE, DISTSMetric, LPIPSMetric, SAMMetric, ARNIQAMetric

In [ ]:
from prettytable import PrettyTable

def get_model_params(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        try:
            params = parameter.numel()
            table.add_row([name, params])
            total_params += params
        except Exception as e:
            print(f"Error processing parameter {name}: {e}")
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params

In [ ]:
DECONV1 = {'kern_size': (),
         'max_iters': 100,
         'iso': True}
DECONV2 = {'kern_size': (),
         'max_iters': 100,
         'iso': True}

device = 'cuda'
modelp = Path('D:/Projects/torch-admm-deconv/trained_models/2026/blind/admm/admm-bsd-div2k-15and35std-june-128-cwa-modif-2_epoch04_vloss0.1031.tar')
# FROZE = DivergentRestorer(3, 2, 3,
#                           3, 4, 86,
#                           86, 8,
#                           output_activation=torch.nn.Sigmoid(), admms=[DECONV1, DECONV2])
ADMM = DivergentRestorer([2, 8, 32], 3,
                              3, 86,
                              86, 8,
                              output_activation=torch.nn.Sigmoid(), admms=[DECONV1, DECONV2])
model_d = torch.load(modelp, weights_only=False)
ADMM.load_state_dict(model_d['model_state_dict'])
ADMM = ADMM.to(device)
ADMM = ADMM.eval()

In [ ]:
get_graph_node_names(ADMM)

In [ ]:
feature_extractor = create_feature_extractor(ADMM, return_nodes=['blocks.0.admms.0', 'blocks.0.admms.1'])

In [ ]:
min_std, max_std = 25, 26
xp = Path(r'D:\\Projects\\datasets\\denoising_test_sets\\CBSD68')
yp = xp

In [ ]:
imd = ImageDataset(xp, yp, transforms=[Scale(), AddAWGN(std_range=(min_std, max_std), both=False)])
im_loader = torch.utils.data.DataLoader(imd, shuffle=False, batch_size=1)

In [ ]:
x_test, y_test = next(iter(im_loader))
features = feature_extractor(x_test.to(device))
feat_admm0 = features['blocks.0.admms.0']
feat_admm1 = features['blocks.0.admms.1']

In [ ]:
# Vizualize the rgb image
plt.figure(figsize=(10, 10))
plt.imshow(feat_admm0.detach().cpu().numpy()[0].transpose(1, 2, 0))

In [ ]:
# Vizualize the rgb image
plt.figure(figsize=(10, 10))
plt.imshow(feat_admm1.detach().cpu().numpy()[0].transpose(1, 2, 0))